# CreditGuard AI - Credit Card Approval Preddiction

## Part 1: Data Cleaning and Exploratory Data Analysis

This notebook performs complete data preparation and analysis on the Credit Card Approval Prediction dataset.

Objectives:

- Load and inspect raw customer data
- Identify missing values
- Handle incorrect data types
- Remove duplicates
- Analyze statistical properties
- Detect skewness and outliers
- Create visualizations
- Study feature relationships

The cleaned dataset will be used for Machine Learning model development.

## 1. Import Required Libraries

Pandas and NumPy are used for data processing.

Matplotlib and Seaborn are used for data visualization.

In [ ]:
import pandas as pd
import numpy as np

import matplotlib.pyplot as plt
import seaborn as sns

import os
import warnings

warnings.filterwarnings("ignore")


os.makedirs(
    "plots",
    exist_ok=True
)

print("Libraries imported successfully")

## 2. Load Dataset

The raw credit card customer dataset is loaded using pandas read_csv().


In [ ]:
df = pd.read_csv("/content/application_record.csv")
df.head()

## 3. Initial Data Inspection

Before cleaning, we inspect:

- First rows
- Column data types
- Dataset dimensions

In [ ]:
print("Dataset Shape")
print(df.shape)

print("\nData Types")
print(df.dtypes)

## 4. Null Value Analysis

Missing values are identified using:

- Missing count
- Missing percentage

Columns above 20% missing values are reported separately.

In [ ]:
null_count = df.isnull().sum()
null_percentage = (null_count/df.shape[0])*100

null_table = pd.DataFrame({"Missing Count":null_count,"Missing Percentage":null_percentage})
null_table

In [ ]:
above_20 = null_table[null_table["Missing Percentage"]>20]
above_20

## Missing Value Treatment

For numeric columns with less than 20% missing data, median imputation is used.

Median is selected because financial data contains extreme values and median is less affected by outliers compared with mean.

For columns with more than 20% missing data , the column is dropped.

In [ ]:
numeric_cols = df.select_dtypes(include=["int64","float64"]).columns


for col in numeric_cols:
    if df[col].isnull().mean()<0.20:
        df[col]=df[col].fillna(df[col].median())


df.isnull().sum()

In [ ]:
df.drop(above_20.index,axis=1,inplace=True)
df.isnull().sum()

## 5. Duplicate Detection and Removal

In [ ]:
duplicates = df.duplicated().sum()


print("Duplicate Rows:",duplicates)
before=df.shape[0]
df=df.drop_duplicates()
after=df.shape[0]
print("Rows Removed:",before-after)

## 6. Data Type Correction and Memory Optimization

Categorical variables are converted from object datatype into category datatype to reduce memory usage.

In [ ]:
before_memory = (df.memory_usage(deep=True).sum())


print("Memory Before:",before_memory)

df["NAME_EDUCATION_TYPE"] = (df["NAME_EDUCATION_TYPE"].astype("category"))
after_memory = (df.memory_usage(deep=True).sum())


print("Memory After:",after_memory)

## 7. Feature Engineering

DAYS_BIRTH contains negative day values.

It is converted into customer AGE.


Removing Unusual Values in DAYS_EMPLOYEED

In [ ]:
df["AGE"] = (abs(df["DAYS_BIRTH"])/365).astype(int)
df.head()

# ============================================
# Fix DAYS_EMPLOYED
# ============================================


# Check unusual values

print(df["DAYS_EMPLOYED"].describe())
print("365243 Count:",(df["DAYS_EMPLOYED"] == 365243).sum())

# Replace invalid employment value with 0
df["DAYS_EMPLOYED"] = (df["DAYS_EMPLOYED"].replace(365243,0))


# Convert days into employment years
df["EMPLOYMENT_YEARS"] = (abs(df["DAYS_EMPLOYED"])/365)

df[["DAYS_EMPLOYED","EMPLOYMENT_YEARS"]].head()

## 8. Descriptive Statistics

In [ ]:
df.drop(columns=["DAYS_BIRTH","DAYS_EMPLOYED"],axis=1,inplace=True)
df.head()


In [ ]:
df.describe()

## 9. Skewness Analysis

Skewness explains the symmetry of data distribution.

Positive skew:
Extreme high values pull the mean upward.

Negative skew:
Extreme low values pull the mean downward.

In [ ]:
numeric_cols=df.select_dtypes(include=["int64","float64"]).columns

skew_values={}
for col in numeric_cols:
    skew_values[col]=df[col].skew()
skew_values

In [ ]:
most_skewed=max(skew_values,key=lambda x:abs(skew_values[x]))

print("Highest Skew Column:",most_skewed)
print("Value:",skew_values[most_skewed])

## 10. IQR Outlier Detection

The Inter Quartile Range (IQR) method is used to detect extreme values.

Formula:

IQR = Q3 - Q1

Lower Bound = Q1 - 1.5 × IQR

Upper Bound = Q3 + 1.5 × IQR


Outliers are detected but not removed because extreme income values may represent real customers.

In [ ]:
def detect_outliers(column):
    Q1 = df[column].quantile(0.25)
    Q3 = df[column].quantile(0.75)
    IQR = Q3 - Q1

    lower = Q1 - 1.5 * IQR
    upper = Q3 + 1.5 * IQR

    outliers = df[(df[column] < lower)|(df[column] > upper)]
    print("Column:", column)
    print("Q1:", Q1)
    print("Q3:", Q3)
    print("IQR:", IQR)
    print("Lower Bound:", lower)
    print("Upper Bound:", upper)
    print("Outlier Count:",len(outliers))
    print("-"*40)

detect_outliers("AMT_INCOME_TOTAL")
detect_outliers("AGE")

# 11. Data Visualization

Five visualizations are created:

1. Line plot
2. Bar chart
3. Histogram
4. Scatter plot
5. Box plot


## Line Plot

Shows customer age trend across dataset records.

In [ ]:
plt.figure(figsize=(10,5))
plt.plot(df["AGE"])
plt.title("Customer Age Trend")
plt.xlabel("Customer Index")
plt.ylabel("Age")
plt.savefig("plots/age_line_plot.png")
plt.show()

## Bar Chart

This compares average income across different education categories.

In [ ]:
education_income = (df.groupby("NAME_EDUCATION_TYPE")["AMT_INCOME_TOTAL"].mean())
plt.figure(figsize=(10,5))
education_income.plot(kind="bar")
plt.title("Average Income by Education Level")
plt.xlabel("Education Type")
plt.ylabel("Average Income")
plt.xticks(rotation=45)
plt.savefig("plots/education_income_bar.png")
plt.show()

## Histogram

The most skewed numerical feature is visualized to understand its distribution.

In [ ]:
plt.figure(figsize=(8,5))
sns.histplot(df[most_skewed],bins=10)
plt.title(f"Distribution of {most_skewed}")
plt.xlabel(most_skewed)
plt.ylabel("Frequency")
plt.savefig("plots/skew_histogram.png")
plt.show()

## Scatter Plot

The relationship between customer age and income is analyzed.

In [ ]:
plt.figure(figsize=(8,5))
sns.scatterplot(data=df,x="AGE",y="AMT_INCOME_TOTAL")
plt.title("Age vs Income Relationship")
plt.xlabel("Age")
plt.ylabel("Income")
plt.savefig("plots/age_income_scatter.png")
plt.show()

## Box Plot

The boxplot compares income distribution among education categories.

In [ ]:
plt.figure(figsize=(12,5))
sns.boxplot(data=df,x="NAME_EDUCATION_TYPE",y="AMT_INCOME_TOTAL")
plt.xticks(rotation=45)
plt.title("Income Distribution by Education")
plt.xlabel("Education")
plt.ylabel("Income")
plt.savefig("plots/income_boxplot.png")
plt.show()

# 12. Pearson Correlation Heatmap

Pearson correlation measures the linear relationship between numerical variables.

Values range:

+1 = strong positive relationship

0 = no relationship

-1 = strong negative relationship

In [ ]:
# Select numeric columns only
numeric_df = df.select_dtypes(include=["int64","float64"])

# Pearson correlation
pearson_corr = numeric_df.corr()

print("Pearson Correlation Matrix")
pearson_corr

In [ ]:
plt.figure(figsize=(12,8))
sns.heatmap(pearson_corr,annot=True,fmt=".2f")
plt.title("Pearson Correlation Heatmap")
plt.savefig("plots/correlation_heatmap.png")
plt.show()

## Highest Correlation Pair

The strongest relationship between two numeric variables is identified.

In [ ]:
corr_pairs = (pearson_corr.abs().unstack().sort_values(ascending=False))

# Remove self correlation
corr_pairs = corr_pairs[corr_pairs < 1]
highest_corr = corr_pairs.head(1)

print("Highest Correlation Pair:")
print(highest_corr)

The highest correlated variables were identified using Pearson correlation.

Correlation does not always indicate causation.

A third factor such as employment status, education, or financial background may influence both variables.}

# 13. Mean vs Median Comparison for Skewed Columns

The two highest skewed columns are analyzed.

Mean and median values are compared before choosing an imputation strategy.

In [ ]:
top_two_skewed = (pd.Series(skew_values).abs().sort_values(ascending=False).head(2).index)


for col in top_two_skewed:

    print("\nColumn:",col)
    print("Mean:",df[col].mean())


    print("Median:",df[col].median())

In [ ]:
for col in top_two_skewed:

    df[col] = (df[col].fillna(df[col].median()))

print("Remaining Null Values")
df.isnull().sum()

Median was selected for imputation.

For positively skewed columns, extreme high values pull the mean upward.

For negatively skewed columns, extreme low values pull the mean downward.

Median gives a more reliable central value.

# 14. Spearman Rank Correlation

Spearman correlation measures monotonic relationships.

Unlike Pearson, it works on ranks and can identify non-linear relationships.

In [ ]:
spearman_corr = df.corr(method="spearman",numeric_only=True)

print("Spearman Correlation Matrix")
spearman_corr

In [ ]:
difference = abs(spearman_corr-pearson_corr)

print("Difference Matrix")
difference

In [ ]:
top_three_difference = (difference.unstack().sort_values(ascending=False).head(3))

print("Top 3 Pearson vs Spearman Differences")

print(top_three_difference)

The largest differences between Pearson and Spearman correlations were analyzed.

If Spearman is higher than Pearson, the relationship is monotonic but not perfectly linear.

For feature selection, both methods will be considered because financial data often contains non-linear relationships.

# 15. Grouped Aggregation Analysis

Education category is compared with income.

Mean, standard deviation and record count are calculated.

In [ ]:
group_analysis = (df.groupby("NAME_EDUCATION_TYPE")["AMT_INCOME_TOTAL"].agg(["mean","std","count"]))
group_analysis

In [ ]:
highest_mean = (group_analysis["mean"].idxmax())
highest_std = (group_analysis["std"].idxmax())

ratio = (group_analysis["mean"].max()/group_analysis["mean"].min())

print("Highest Mean Group:",highest_mean)
print( "Highest STD Group:",highest_std)
print("Mean Ratio:",ratio)

The group with highest mean income was identified.

A high standard deviation indicates large variation inside that group.

This means education alone cannot fully predict customer income.

Other features are required for accurate prediction.

# 16. Save Clean Dataset

The final processed dataset is saved for Machine Learning.

In [ ]:
df.to_csv(
"cleaned_credit_data.csv",index=False)

print("Clean dataset saved successfully")